# HEN Final Experiment Notebook

This notebook is the executable companion to the final Flat CNN vs HEN report. It keeps the long-running training commands in one place, but it also embeds the final local results directly in the notebook output so readers do not need to retrain every model just to inspect the comparison.

**Default behavior:** no long training is started. The notebook shows the final measured results first, then provides optional commands for reproducing each experiment.

## 1. Environment setup

If this notebook is opened directly in Colab, the first cell clones the GitHub repository. If it is already running inside the repository, it simply uses the current directory.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import textwrap

REPO_URL = "https://github.com/HengchunSong/HEN.git"
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "train_joint_hen.py").exists():
    colab_dir = Path("/content/HEN")
    if colab_dir.exists():
        subprocess.run(["git", "-C", str(colab_dir), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(colab_dir)], check=True)
    os.chdir(colab_dir)
    PROJECT_DIR = colab_dir

print("Notebook ready. Long training is disabled by default.")
print("Expected repository:", REPO_URL)

Notebook ready. Long training is disabled by default.
Expected repository: https://github.com/HengchunSong/HEN.git


In [ ]:
# Colab-only dependency cell. It is safe to run locally too.
# The project training scripts require PyTorch and torchvision, which Colab GPU runtimes already include.
!pip install -q pandas matplotlib

## 2. The 3-9-27 hierarchy

The final experiments use a balanced 27-class ImageNet subset arranged as a three-level tree: 3 coarse groups, 9 mid-level groups, and 27 leaf classes.

In [2]:
HIERARCHY = {
    "level1": ["animal", "vehicle", "food"],
    "level2": [
        "bird", "canine", "feline",
        "aircraft", "road_vehicle", "watercraft",
        "fruit", "vegetable", "prepared_food",
    ],
    "leaf": [
        "goldfinch", "robin", "bald_eagle",
        "golden_retriever", "german_shepherd", "doberman",
        "persian_cat", "siamese_cat", "egyptian_cat",
        "airliner", "airship", "warplane",
        "ambulance", "fire_engine", "sports_car",
        "canoe", "container_ship", "speedboat",
        "lemon", "banana", "pomegranate",
        "broccoli", "cauliflower", "cucumber",
        "cheeseburger", "hotdog", "pizza",
    ],
}

print(f"Level 1 groups: {len(HIERARCHY['level1'])} -> {HIERARCHY['level1']}")
print(f"Level 2 groups: {len(HIERARCHY['level2'])} -> {HIERARCHY['level2']}")
print(f"Leaf classes: {len(HIERARCHY['leaf'])}")

Level 1 groups: 3 -> ['animal', 'vehicle', 'food']
Level 2 groups: 9 -> ['bird', 'canine', 'feline', 'aircraft', 'road_vehicle', 'watercraft', 'fruit', 'vegetable', 'prepared_food']
Leaf classes: 27


## 3. Final local results embedded from the report

The table below is copied from the local experiment summaries used by the final report. These are the values to inspect when training time is too expensive to repeat.

In [3]:
FINAL_RESULTS = [
    {
        "method": "Flat ConvNeXt-Tiny",
        "family": "Flat",
        "backbone": "ConvNeXt-Tiny",
        "accuracy_pct": 96.8889,
        "params_m": "27.84",
        "compute_gflops": "4.456",
        "status": "best absolute accuracy",
        "summary_path": "outputs/flat27_convnext_tiny_run1/summary.json",
    },
    {
        "method": "Flat ResNet18 + SAM",
        "family": "Flat",
        "backbone": "ResNet18",
        "accuracy_pct": 94.9630,
        "params_m": "11.19",
        "compute_gflops": "1.814",
        "status": "close to 95%, below target",
        "summary_path": "outputs/flat27_resnet18_sam_run1/summary.json",
    },
    {
        "method": "Flat MobileNetV3-Small",
        "family": "Flat",
        "backbone": "MobileNetV3-Small",
        "accuracy_pct": 94.5185,
        "params_m": "1.55",
        "compute_gflops": "0.057",
        "status": "too small to cross 95%",
        "summary_path": "outputs/flat27_mobilenet_v3_small_run1_20260405/summary.json",
    },
    {
        "method": "Classic 3-Level HEN",
        "family": "HEN classic",
        "backbone": "ResNet18 routers and experts",
        "accuracy_pct": 95.6296,
        "params_m": "145.31 total / 33.53 active",
        "compute_gflops": "5.442 active",
        "status": "accurate but storage-heavy",
        "summary_path": "outputs/hen_eval_summary_food_3level_feline_aircraft_improved.json",
    },
    {
        "method": "Coarse-to-Fine HEN full",
        "family": "HEN c2f",
        "backbone": "ShuffleNetV2 router + ResNet18",
        "accuracy_pct": 92.3704,
        "params_m": "36.23",
        "compute_gflops": "~0.64 active estimate",
        "status": "did not reach leaf target",
        "summary_path": "outputs/c2f_hen_full_shufflenetx10_r128_residual_ft1_resnet18_20260404/summary.json",
    },
    {
        "method": "Joint HEN MobileNetV3-Large",
        "family": "HEN compact",
        "backbone": "MobileNetV3-Large",
        "accuracy_pct": 95.6296,
        "params_m": "3.01",
        "compute_gflops": "0.217",
        "status": "best efficiency tradeoff",
        "summary_path": "outputs/joint_hen_mobilenet_v3_large_run1_20260430/summary.json",
    },
    {
        "method": "Selective Common-Delta HEN",
        "family": "HEN research",
        "backbone": "MobileNetV3-Small",
        "accuracy_pct": 93.7778,
        "params_m": "1.74 total / 0.79 trained",
        "compute_gflops": "0.057 backbone-only",
        "status": "useful locally, not winner",
        "summary_path": "outputs/common_delta_hen_mobilenet_v3_small_selective_fap_run1_20260430/summary.json",
    },
]

columns = ["method", "family", "backbone", "accuracy_pct", "params_m", "compute_gflops", "status"]
widths = {column: max(len(column), *(len(str(row[column])) for row in FINAL_RESULTS)) for column in columns}
for column in columns:
    print(str(column).ljust(widths[column]), end="  ")
print()
for row in FINAL_RESULTS:
    for column in columns:
        value = f"{row[column]:.4f}" if isinstance(row[column], float) else str(row[column])
        print(value.ljust(widths[column]), end="  ")
    print()

method                              family        backbone                         accuracy_pct  params_m                       compute_gflops            status
Flat ConvNeXt-Tiny                  Flat          ConvNeXt-Tiny                    96.8889       27.84                          4.456                    best absolute accuracy
Flat ResNet18 + SAM                 Flat          ResNet18                         94.9630       11.19                          1.814                    close to 95%, below target
Flat MobileNetV3-Small              Flat          MobileNetV3-Small                94.5185       1.55                           0.057                    too small to cross 95%
Classic 3-Level HEN                 HEN classic   ResNet18 routers and experts     95.6296       145.31 total / 33.53 active    5.442 active             accurate but storage-heavy
Coarse-to-Fine HEN full             HEN c2f       ShuffleNetV2 router + ResNet18   92.3704       36.23                         

## 4. Resource-efficiency calculation

The main comparison is not whether HEN beats the strongest flat CNN in absolute accuracy. It does not. The key result is that compact Joint HEN crosses the practical 95% line with a much smaller resource budget.

In [4]:
flat_acc = 96.8889
flat_params = 27.84
flat_gflops = 4.456
hen_acc = 95.6296
hen_params = 3.01
hen_gflops = 0.217

print("Flat ConvNeXt-Tiny vs Compact Joint HEN:")
print(f"- Accuracy gap: {flat_acc - hen_acc:.2f} percentage points")
print(f"- Parameter ratio: {flat_params / hen_params:.2f}x more parameters for flat ConvNeXt-Tiny")
print(f"- Compute ratio: {flat_gflops / hen_gflops:.2f}x more GFLOPs for flat ConvNeXt-Tiny")
print(
    f"- Compact HEN keeps {hen_acc / flat_acc * 100:.2f}% of the flat ConvNeXt accuracy "
    f"using {hen_params / flat_params * 100:.2f}% of the parameters and "
    f"{hen_gflops / flat_gflops * 100:.2f}% of the compute."
)

Flat ConvNeXt-Tiny vs Compact Joint HEN:
- Accuracy gap: 1.26 percentage points
- Parameter ratio: 9.25x more parameters for flat ConvNeXt-Tiny
- Compute ratio: 20.53x more GFLOPs for flat ConvNeXt-Tiny
- Compact HEN keeps 98.70% of the flat ConvNeXt accuracy using 10.81% of the parameters and 4.87% of the compute.


In [ ]:
# Optional visualization. Run this cell to regenerate the accuracy/parameter/compute chart.
import matplotlib.pyplot as plt

plot_rows = [
    ("Flat ConvNeXt", 96.8889, 27.84, 4.456),
    ("Classic HEN", 95.6296, 145.31, 5.442),
    ("Compact Joint HEN", 95.6296, 3.01, 0.217),
    ("Common-Delta", 93.7778, 1.74, 0.057),
]
labels = [row[0] for row in plot_rows]
accuracy = [row[1] for row in plot_rows]
params = [row[2] for row in plot_rows]
compute = [row[3] for row in plot_rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(labels, accuracy)
axes[0].axhline(95, color="tab:red", linestyle="--", linewidth=1)
axes[0].set_title("Accuracy (%)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(labels, params)
axes[1].set_title("Parameters (M)")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(labels, compute)
axes[2].set_title("Compute (GFLOPs)")
axes[2].tick_params(axis="x", rotation=25)

fig.tight_layout()
plt.show()

## 5. Load local summaries if they exist

When this notebook is run inside the repository after the experiments have been generated, the cell below reads the same `summary.json` files used by the report. If the files are missing, the embedded table above remains the source of truth.

In [5]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

loaded = {}
for row in FINAL_RESULTS:
    loaded[row["method"]] = read_json(row["summary_path"])

for method, data in loaded.items():
    if data is None:
        continue
    if "best_val_acc_pct" in data:
        print(f"{method}: loaded best_val_acc_pct={data['best_val_acc_pct']}")
    elif "best_leaf_val_acc_pct" in data:
        print(f"{method}: loaded best_leaf_val_acc_pct={data['best_leaf_val_acc_pct']}")
    elif "end_to_end_leaf_acc" in data:
        print(f"{method}: loaded end_to_end_leaf_acc={data['end_to_end_leaf_acc'] * 100:.4f}")
    elif "best_score_pct" in data:
        print(f"{method}: loaded best_score_pct={data['best_score_pct']}")

print("Summary file check complete.")
print("Embedded report values remain available even when local summary files are absent.")

Summary file check complete.
Embedded report values remain available even when local summary files are absent.


## 6. Reproduction commands

The following commands are included for reproducibility. They can take many hours on a single GPU, so they are printed rather than executed by default.

In [6]:
TRAINING_COMMANDS = {
    "flat_convnext_tiny": "python train_flat_27.py --data-root data/imagenet_subset_food --arch convnext_tiny --epochs 20 --batch-size 96 --image-size 224 --output-dir outputs/flat27_convnext_tiny_run1 --channels-last",
    "flat_mobilenet_v3_small": "python train_flat_27.py --data-root data/imagenet_subset_food --arch mobilenet_v3_small --epochs 20 --batch-size 96 --image-size 224 --output-dir outputs/flat27_mobilenet_v3_small_run1_20260405 --channels-last",
    "joint_hen_mobilenet_v3_large": "python train_joint_hen.py --data-root data/imagenet_subset_food --backbone mobilenet_v3_large --epochs 25 --batch-size 96 --image-size 224 --output-dir outputs/joint_hen_mobilenet_v3_large_run1_20260430 --channels-last",
    "coarse_to_fine_full": "python train_c2f_hen.py --data-root data/imagenet_subset_food --backbone resnet18 --router-backbone shufflenet_v2_x1_0 --image-size 128 --router-image-size 128 --scope full --epochs 12 --batch-size 96 --output-dir outputs/c2f_hen_full_shufflenetx10_r128_residual_ft1_resnet18_20260404 --channels-last",
    "common_delta_selective": "python train_common_delta_hen.py --data-root data/imagenet_subset_food --backbone mobilenet_v3_small --epochs 25 --batch-size 96 --image-size 224 --split-hidden-dim 64 --delta-dim 32 --common-delta-level2 feline aircraft prepared_food --split-entropy-weight 0.002 --output-dir outputs/common_delta_hen_mobilenet_v3_small_selective_fap_run1_20260430 --channels-last",
}

for name, command in TRAINING_COMMANDS.items():
    print(f"[{name}]\n{command}\n")

[flat_convnext_tiny]
python train_flat_27.py --data-root data/imagenet_subset_food --arch convnext_tiny --epochs 20 --batch-size 96 --image-size 224 --output-dir outputs/flat27_convnext_tiny_run1 --channels-last

[flat_mobilenet_v3_small]
python train_flat_27.py --data-root data/imagenet_subset_food --arch mobilenet_v3_small --epochs 20 --batch-size 96 --image-size 224 --output-dir outputs/flat27_mobilenet_v3_small_run1_20260405 --channels-last

[joint_hen_mobilenet_v3_large]
python train_joint_hen.py --data-root data/imagenet_subset_food --backbone mobilenet_v3_large --epochs 25 --batch-size 96 --image-size 224 --output-dir outputs/joint_hen_mobilenet_v3_large_run1_20260430 --channels-last

[coarse_to_fine_full]
python train_c2f_hen.py --data-root data/imagenet_subset_food --backbone resnet18 --router-backbone shufflenet_v2_x1_0 --image-size 128 --router-image-size 128 --scope full --epochs 12 --batch-size 96 --output-dir outputs/c2f_hen_full_shufflenetx10_r128_residual_ft1_resnet18_2

In [7]:
RUN_LONG_TRAINING = False
SELECTED_EXPERIMENT = "joint_hen_mobilenet_v3_large"

if RUN_LONG_TRAINING:
    command = TRAINING_COMMANDS[SELECTED_EXPERIMENT]
    print("Running:", command)
    subprocess.run(command.split(), check=True)
else:
    print("Long training skipped. Set RUN_LONG_TRAINING=True and choose SELECTED_EXPERIMENT to reproduce one run.")

Long training skipped. Set RUN_LONG_TRAINING=True and choose SELECTED_EXPERIMENT to reproduce one run.


## 7. Final interpretation

- The strongest absolute model remains **Flat ConvNeXt-Tiny** at **96.89%**.
- The best compact answer is **Joint HEN + MobileNetV3-Large** at **95.63%**, using **3.01M parameters** and about **0.217 GFLOPs**.
- The 95% target is practical because the validation set contains ambiguous mixed-object images, so the last percentage point is partly architecture and partly data-label noise.
- Coarse-to-Fine and Common-Delta were useful research directions, but neither became the final efficiency/accuracy winner.